**Import Libraries**

In [11]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import LSTM, Dense, Embedding, TextVectorization
from tensorflow.keras.models import Sequential

**Load the Dataset (Fake.csv and True.csv from Kaggle)**

In [12]:
fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

In [13]:
fake["label"] = 0  # 0 = Fake News
true["label"] = 1  # 1 = Real News

**Combine both datasets and shuffle the rows**

In [14]:
df = pd.concat([fake, true]).sample(frac=1, random_state=42)
texts = (df["title"] + " " + df["text"]).astype(str).values
labels = df["label"].values

**Split the Dataset into Training and Testing Sets**

In [15]:
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42)

**Convert words into integer sequences automatically**

In [16]:
vectorizer = TextVectorization(max_tokens=10000, output_sequence_length=200)
vectorizer.adapt(X_train)

**Build the Deep Learning Model (Embedding + LSTM)**

In [17]:
model = Sequential(
    [
        vectorizer,
        Embedding(input_dim=10000, output_dim=64),
        LSTM(64),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),  # Outputs probability between 0 and 1
    ]
)

**Compile the Model**

In [18]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

**Train the model**

In [19]:
print("Starting training...")
model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test))

Starting training...
Epoch 1/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 92s 160ms/step - accuracy: 0.8066 - loss: 0.4563 - val_accuracy: 0.5614 - val_loss: 0.6598
Epoch 2/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 89s 159ms/step - accuracy: 0.8282 - loss: 0.4143 - val_accuracy: 0.8403 - val_loss: 0.4076
Epoch 3/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 98s 175ms/step - accuracy: 0.8727 - loss: 0.3315 - val_accuracy: 0.8892 - val_loss: 0.3208
Epoch 4/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 134s 162ms/step - accuracy: 0.9550 - loss: 0.1439 - val_accuracy: 0.9688 - val_loss: 0.1005
Epoch 5/5
562/562 ━━━━━━━━━━━━━━━━━━━━ 141s 161ms/step - accuracy: 0.8838 - loss: 0.2580 - val_accuracy: 0.6967 - val_loss: 0.4822


**Test the model on custom news headlines**

In [20]:
def check_news(news_text):
    prob = model.predict(tf.constant([news_text]))[0][0]
    if prob > 0.5:
        print(f"Result: REAL News ({prob * 100:.1f}% confidence)")
    else:
        print(f"Result: FAKE News ({(1 - prob) * 100:.1f}% confidence)")

**Save the model**

In [22]:
model.save("fake_news_model.keras")
print("Model saved successfully as fake_news_model.keras")

Model saved successfully as fake_news_model.keras
